# Ultraphenomics

Questo Colab è incentrato sul come si può mettere in piedi una pipeline *end-to-end* di elaborazione dei dati. Una pipeline di questo tipo si basa normalmente su una sequenza di questo tipo:

1. *Raccolta dei dati*
2. *Creazione del dataset*
3. *Addestramento della rete*
4. *Interpretazione dei risultati*

Vediamo adesso più nel dettaglio ciascuna di queste fasi.

### 1. Raccolta dei dati

Gran parte della riuscita di un esperimento di machine learning dipende dalla corretta raccolta dei dati. In tal senso, è necessario progettare adeguatamente tale esperimento, considerando non solo il tipo di dato da raccogliere, ma anche fattori come l'influenza del meteo, le caratteristiche dei sensori scelti, il tipo di dato, la riproducibilità dell'esperimento e la qualità delle acquisizioni. Ovviamente, la qualità della campagna di acquisizione influenza direttamente le potenzialità del modello: migliori saranno i dati raccolti, infatti, maggiori saranno le performance e, potenzialmente, l'efficacia dello stesso.

### 2. Creazione del dataset

Dopo aver completato la raccolta dei dati è necessario organizzarli in un formato "digeribile" dai nostri modelli di machine learning; in altre parole, dovremo *creare il dataset*.

La creazione del dataset può seguire diverse strade, a seconda delle finalità e del livello di "standardizzazione" richiesto. Tuttavia, molto spesso è necessario *etichettare* il dataset, in maniera da fornire una certa "conoscenza di dominio" al modello. Per farlo potremmo scegliere anche strade "alternative", come abbiamo visto nella lezione precedente, ed utilizzare un metodo automatico per fornire una prima etichettatura da "sgrossare" successivamente.

E' tuttavia importante sottolineare come, in generale, sia necessario porre alcuni accorgimenti per fare in modo che il dataset sia adeguato ad affrontare il task che ci siamo prefissati. In tal senso, facciamo un rapido esempio basato su un modello di deep learning per la object detection.

Supponendo di voler addestrare un'architettura come YOLO nelle versioni recenti *da zero*, avremo bisogno di un quantitativo di immagini abbastanza rilevante: non c'è un numero minimo prefissato, ma di solito si considerano come soglia minima le $1500$ immagini con $10000$ bounding box o maschere di segmentazione per ogni classe. Questo fa subito emergere come contingente il problema della *scarsità di dati*, particolarmente rilevante in campi come l'agronomia, che sono spesso condizionati dai cicli stagionali e dalla naturale evoluzione dei tratti fenotipici delle piante. Tuttavia, fatte queste opportune considerazioni, potremo passare alla fase successiva, ovvero all'uso di questo dataset per *addestrare il modello*.

### 3. Addestramento del modello

Abbiamo detto che i modelli devono essere addestrati a partire dal dataset (etichettato o meno) la cui creazione è l'obiettivo del passo precedente. Supponiamo, come al solito, di stare utilizzando un dataset per la object detection, e di voler creare un modello di deep learning basato su un'architettura YOLO.

Per prima cosa, dovremo importare la classe `YOLO` dal package `ultralytics`:

In [ ]:
from ultralytics import YOLO

A questo punto dovremo istanziare un modello, non utilizzando i pesi preaddestrati, bensì quelli "allenabili" contenuti nel file in formato `yaml`:

In [ ]:
model = YOLO('yolo26n.yaml')

Istanziato il modello, dovremo chiamare il metodo `train`, specificando il percorso dove si trova il dataset e, opzionalmente, una importante serie di parametri. Ad esempio:

In [ ]:
results = model.train(
    data='data',
    epochs=100,
    seed=42)

In questo caso, stiamo passando tre parametri, di cui uno fondamentale. In particolare:

* il parametro `data` indica dove il percorso fisico del dataset, ovvero la cartella in cui avremo le immagini e le etichette associate;
* il parametro `epochs`, che di default vale 300, indica il numero di epoche di addestramento della rete;
* il parametro `seed` va impostato ad un valore a nostro piacimento per permettere la *riproducibilità*, dell'esperimento.

> NOTA: L'elenco dei parametri che è possibile impostare, assieme ai valori di default, è disponibile a [questo indirizzo](https://docs.ultralytics.com/modes/train/#train-settings).

### 3. Interpretare i risultati

Una volta completata la procedura di addestramento è nostro compito valutare in maniera sia *qualitativa* sia *quantitativa* i risultati ottenuti. Cerchiamo di capire come sia possibile operare su entrambe queste direttrici.

#### Interpretazione quantitativa

L'interpretazione quantitativa dei risultati ottenuti è quella più semplice da portare avanti, in quanto non richiede particolari accorgimenti o studi, ma necessita esclusivamente della valutazione numerica di alcune metriche. Queste sono generate in automatico dalla libreria Ultralytics durante l'addestramento del modello, e vengono salvate all'interno della cartella dei risultati nel file `results.csv`. Di particolare interesse sono tre valori, ovvero:

* *precisione*, che rappresenta (in percentuale) la capacità del modello addestrato di localizzare e classificare correttamente gli oggetti di interesse all'interno di un'immagine;
* *recall*, che rappresenta (in percentuale) l'abilità del modello di caratterizzare tutta l'informazione di interesse nella scena (e, quindi, di non "perdere per strada" oggetti di interesse);
* *mean average precision*, rappresentativa della precisione con cui il modello va a posizionale le bounding box all'interno dell'immagine analizzata.

Queste tre metriche sono espresse poi al variare di un certo *livello di confidenza*, che rappresenta il "grado di certezza" del modello nello stabilire che una certa porzione di immagine è relativa proprio ad un certo oggetto desiderato.

Infine, il grado di accuratezza complessiva del modello è espresso da una *matrice di confusione*, che mette in relazione le predizioni del modello con il *ground truth*, ovvero la verità "vera". Partiamo da quest'ultima.

##### La matrice di confusione

Nella figura successiva mostriamo una matrice di confusione derivante da un esperimento di plant phenotyping pubblicato lo scorso anno. Cerchiamo di interpretarla semplicemente osservandone la struttura.

<img src="./images/confusion_matrix.png" width="405">

Notiamo subito che sull'asse orizzontale abbiamo i valori *True*, ovvero quelli relativi al ground truth, mentre sull'asse verticale possiamo vedere i valori *Predicted*, ovvero le predizioni mandate in output dalla rete. Per quello che riguarda le etichette, abbiamo sia sull'asse orizzontale, sia su quello verticale, quattro diverse diciture, ovvero *fruit*, *node*, *flower* e *background*.

Ora, queste diciture sono normalmente associate alle *classi* presenti nel dataset, per cui possiamo dire che nel dataset originario abbiamo una classe *fruit*, una classe *node* ed una classe *flower*; per quello che riguarda invece il *background*, questo è presente soltanto nelle matrici di confusione per la object detection, e ne discuteremo più ampiamente a breve. Per adesso, guardiamo le prime tre righe e tre colonne della matrice: per comprendere ciò che indicano, dovremo interpretarla guardando la combinazione tra *predicted* e *true*. In altre parole:

* la rete ha predetto come *fruit* $334$ campioni che, effettivamente, erano *fruit* (riga 1, colonna 1);
* la rete ha predetto come *fruit* $2$ campioni che erano invece *node* (riga 1, colonna 2);
* la rete ha predetto come *node* $2$ campioni che erano invece *fruit* (riga 2, colonna 1);
* la rete ha predetto come *node* $1514$ campioni che, effettivamente, erano *node* (riga 2, colonna 2)...

...e via dicendo. In linea generale, quindi, una buona matrice di confusione è *quasi diagonale*, il che significa che il modello *predice bene*.

Cosa sono quindi quell'ultima riga e colonna con l'etichetta *background*? Cerchiamo di capirlo utilizzando lo stesso principio applicato in precedenza, e vediamo che:

* dall'ultima riga, la rete ha predetto come *background* diverse istanze di *fruit*, *node* e *flower*;
* dall'ultima colonna, la rete ha predetto come *fruit*, *node* o *flower* diverse istanze di *background*.

Ok, ma cosa significa? Per capirlo dobbiamo prima definire il concetto di *background* come zona dell'immagine *non occupata da oggetti di interesse*. In altre parole:

* se la rete predice come *background* una zona in cui abbiamo un'istanza di *fruit*, *node* o *flower*, la rete sta *mancando la predizione*, generando quindi un *falso negativo*;
* se la rete predice come *fruit*, *node* o *flower* una zona che in realtà è di *background*, la rete sta trovando degli oggetti che non sono stati originariamente etichettati, generando un *falso positivo*.

In definitiva, la matrice di confusione ha questo compito fondamentale: darci una valutazione visiva, impattante ed immediata dell'efficacia del modello nell'effettaure le predizioni.

#### Interpretazione qualitativa

L'interpretazione qualitativa è più complessa, ma ci permette di valutare in maniera efficace il *perché* un modello effettua determinate predizioni. Per far questo, possiamo utilizzare le predizioni del modello (che abbiamo visto durante l'esercitazione *You Only Harvest Once*) oppure, in maniera meno intuitiva ma sicuramente più affascinante, le *mappe di attivazione* della rete. Per fare un esempio, usiamo la seguente immagine:

<img src="./images/cam.png" width="450">